In [1]:
import os, re, random, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.base import clone

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
try:
    import lightgbm as lgb
    USE_LGB = True
except ImportError:
    USE_LGB = False
try:
    import xgboost as xgb
    USE_XGB = True
except ImportError:
    USE_XGB = False

warnings.filterwarnings("ignore")



In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Config & Model Zoo
# ─────────────────────────────────────────────────────────────────────────────
SEED              = 42
DATASET_PATH      = "dataset.csv"
FILLED_PATH       = "filled_dataset.csv"
SUBMISSION_PATH   = "submission.csv"

N_SPLITS          = 5
EMBARGO_STEPS     = 2       
HOLDOUT_RATIO     = 0.30    # Used ONLY for offline validation scoring, never training
LAG_WINDOW        = 10      
ROLLING_WINDOWS   = [3, 5]
EWMA_SPAN         = 5
RESIDUAL_WINSOR   = 99.0    
CLIP_PCT_LO       = 0.1
CLIP_PCT_HI       = 99.9
MAX_IV_CAP        = 2.0

random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

MODEL_ZOO = {
    "RandomForest": RandomForestRegressor(
        n_estimators=100, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=150, learning_rate=0.05, max_depth=5, min_samples_leaf=30, random_state=SEED
    )
}

if USE_LGB:
    MODEL_ZOO["LightGBM"] = lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.03, max_depth=5, num_leaves=20, 
        min_child_samples=30, subsample=0.7, colsample_bytree=0.7, 
        reg_alpha=0.1, reg_lambda=5.0, random_state=SEED, verbose=-1
    )
if USE_XGB:
    MODEL_ZOO["XGBoost"] = xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.03, max_depth=5, min_child_weight=30,
        subsample=0.7, colsample_bytree=0.7, reg_lambda=5.0, random_state=SEED, n_jobs=-1
    )

# Removed spline_pred. Features are now strictly independent of the baseline level.
FEATURE_COLS = [
    "log_moneyness", "strike_dist", "is_put",
    "underlying_price", "days_to_expiry", "time_of_day",
    "iv_lag1", "iv_lag2", "iv_lag3",
    "iv_roll3_mean", "iv_roll3_std", "iv_roll5_mean", "iv_roll5_std",
    "iv_ewma",
    "left_neighbor_iv", "right_neighbor_iv", "atm_iv",
]


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ingest, Melt, and Hard Identity (Row IDs)
# ─────────────────────────────────────────────────────────────────────────────
def ingest(path):
    df = pd.read_csv(path)
    for fmt in ["%d-%m-%Y %H:%M", "%Y-%m-%d %H:%M:%S", "%d-%m-%Y %H:%M:%S", None]:
        try:
            df["datetime"] = pd.to_datetime(df["datetime"], format=fmt, dayfirst=True)
            break
        except Exception:
            continue
    df = df.sort_values("datetime").reset_index(drop=True)
    assert df["datetime"].is_monotonic_increasing
    return df

df_wide     = ingest(DATASET_PATH)
df_original = df_wide.copy()
option_cols = [c for c in df_wide.columns if c not in ["datetime","underlying_price"]]

_pat = re.compile(r"NIFTY(\d{2}[A-Z]{3}\d{2})(\d+)(CE|PE)")
def parse_contract(col):
    m = _pat.match(col)
    if not m: return None, None, None
    expiry = pd.to_datetime(m.group(1), format="%d%b%y")
    return int(m.group(2)), m.group(3), expiry

df_long = df_wide.melt(
    id_vars=["datetime","underlying_price"],
    value_vars=option_cols, var_name="option_col", value_name="iv"
)
parsed = df_long["option_col"].apply(
    lambda c: pd.Series(parse_contract(c), index=["strike","option_type","expiry"])
)
df_long = pd.concat([df_long, parsed], axis=1)
df_long = df_long.dropna(subset=["strike","option_type","expiry"])
df_long["strike"] = df_long["strike"].astype(int)
df_long["days_to_expiry"] = (df_long["expiry"] - df_long["datetime"]).dt.days.clip(lower=0)
df_long["time_of_day"] = df_long["datetime"].dt.hour + df_long["datetime"].dt.minute/60.0

# Strict Subset Deduplication & Explicit Row ID Injection
df_long = df_long.sort_values(["datetime","strike"]).reset_index(drop=True)
df_long = df_long.drop_duplicates(subset=["datetime", "strike", "option_type"])
df_long["row_id"] = np.arange(len(df_long))

obs_iv = df_long["iv"].dropna()
DATA_IV_FLOOR = max(0.01, float(np.percentile(obs_iv, 0.1)))
DATA_IV_CAP   = min(MAX_IV_CAP, float(np.percentile(obs_iv, 99.9)))

ts_map = {ts:i for i,ts in enumerate(sorted(df_long["datetime"].unique()))}
df_long["ts_idx"] = df_long["datetime"].map(ts_map)

print(f" Loaded | wide: {df_wide.shape} | long: {df_long.shape} | missing: {df_long['iv'].isna().sum()}")

✅ Loaded | wide: (975, 30) | long: (27300, 11) | missing: 5460


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Static features
# ─────────────────────────────────────────────────────────────────────────────
def add_static(df):
    df = df.copy()
    df["log_moneyness"] = np.log(df["strike"] / df["underlying_price"])
    df["strike_dist"]   = df["strike"] - df["underlying_price"]
    df["is_put"]        = (df["option_type"]=="PE").astype(np.int8)
    df["strike_bucket"] = pd.cut(df["log_moneyness"], bins=10, labels=False)
    return df

df_long = add_static(df_long)


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
#  The Flat-Wing Parabola Builder
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np

def fit_splines(df_block, global_floor, global_cap):
    """
    Fits a 2nd-degree Parabola to get organic, non-zero residuals.
    Stores the min/max log_moneyness boundaries for flat-wing extrapolation.
    """
    splines = {}
    for (ts, otype), grp in df_block.groupby(["datetime","option_type"]):
        obs = grp[grp["iv"].notna()].sort_values("log_moneyness")
        obs = obs.drop_duplicates("log_moneyness")
        x, y = obs["log_moneyness"].values, obs["iv"].values
        
        iv_min = float(y.min()) * 0.5 if len(y) else global_floor
        iv_max = float(y.max()) * 1.5 if len(y) else global_cap
        
        if len(x) >= 3:
            coefs = np.polyfit(x, y, deg=2)
            # Save the x-boundaries alongside the polynomial
            splines[(ts,otype)] = ("poly", np.poly1d(coefs), x.min(), x.max(), iv_min, iv_max)
        elif len(x) == 2:
            coefs = np.polyfit(x, y, deg=1)
            splines[(ts,otype)] = ("poly", np.poly1d(coefs), x.min(), x.max(), iv_min, iv_max)
            
    return splines

def predict_spline_row(row, splines, fallback, gfloor, gcap):
    ts, otype, lm = row["datetime"], row["option_type"], row["log_moneyness"]
    key = (ts, otype)
    
    if key not in splines:
        cands = [(k[0],k) for k in splines if k[1]==otype and k[0]<=ts]
        key = max(cands, key=lambda x:x[0])[1] if cands else None
        if key is None: return fallback
        
    stype, obj, min_x, max_x, iv_min, iv_max = splines[key]
    
    clamped_lm = np.clip(lm, min_x, max_x)
    raw = float(obj(clamped_lm))
    
    return float(np.clip(raw, max(gfloor, iv_min), min(gcap, iv_max)))

def apply_splines(df_block, splines, fallback, gfloor, gcap):
    return np.array([predict_spline_row(r, splines, fallback, gfloor, gcap)
                     for _, r in df_block.iterrows()])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Spatial features (Safe ID Assignment)
# ─────────────────────────────────────────────────────────────────────────────
def build_spatial(df_block):
    df = df_block.copy().set_index("row_id")
    df["left_neighbor_iv"]  = np.nan
    df["right_neighbor_iv"] = np.nan
    df["atm_iv"]            = np.nan
    
    for (ts, otype), grp in df.groupby(["datetime","option_type"]):
        obs = grp[grp["iv"].notna()].sort_values("strike")
        os_, ov_ = obs["strike"].values, obs["iv"].values
        
        for idx, row in grp.iterrows():
            k = row["strike"]; io = not pd.isna(row["iv"])
            sa = os_[os_!=k] if io else os_; va = ov_[os_!=k] if io else ov_
            if len(sa) == 0: continue
            
            lm=sa<k; rm=sa>k
            if lm.any(): df.at[idx,"left_neighbor_iv"]  = va[lm][-1]
            if rm.any(): df.at[idx,"right_neighbor_iv"] = va[rm][0]
            df.at[idx,"atm_iv"] = va[np.argmin(np.abs(sa - row["underlying_price"]))]
            
    for c in ["left_neighbor_iv","right_neighbor_iv","atm_iv"]:
        m = df[c].median()
        df[c] = df[c].fillna(m if not pd.isna(m) else DATA_IV_FLOOR)
        
    return df.reset_index()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Temporal features for TRAINING block
# ─────────────────────────────────────────────────────────────────────────────
def build_temporal_train(df_block):
    df = df_block.copy().sort_values(["datetime","strike"])
    
    def safe_lag(series, n):
        return series.shift(n)
        
    grp = df.groupby(["strike","option_type"])["iv"]

    for lag in [1,2,3]:
        df[f"iv_lag{lag}"] = grp.transform(lambda x: safe_lag(x, lag))

    for w in ROLLING_WINDOWS:
        df[f"iv_roll{w}_mean"] = grp.transform(lambda x: safe_lag(x,1).rolling(w, min_periods=1).mean())
        df[f"iv_roll{w}_std"] = grp.transform(lambda x: safe_lag(x,1).rolling(w, min_periods=1).std())
    df["iv_ewma"] = grp.transform(lambda x: safe_lag(x,1).ewm(span=EWMA_SPAN, adjust=False).mean())

    temporal_cols = ([f"iv_lag{l}" for l in [1,2,3]] +
                     [f"iv_roll{w}_{s}" for w in ROLLING_WINDOWS for s in ["mean","std"]] + ["iv_ewma"])
    train_medians = {c: df[c].median() for c in temporal_cols}
    for c in temporal_cols:
        df[c] = df[c].fillna(train_medians[c])
    return df, train_medians, temporal_cols

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Temporal features for VALIDATION (sequential)
# ─────────────────────────────────────────────────────────────────────────────
def compute_temporal_for_ts(target_rows, history_df, train_medians):
    res = target_rows.copy().set_index("row_id")
    temporal_cols = ([f"iv_lag{l}" for l in [1,2,3]] +
                     [f"iv_roll{w}_{s}" for w in ROLLING_WINDOWS for s in ["mean","std"]] + ["iv_ewma"])
    for c in temporal_cols: res[c] = np.nan

    for (s, ot), grp in res.groupby(["strike","option_type"]):
        hist = (history_df[(history_df["strike"]==s) & (history_df["option_type"]==ot)]
                .sort_values("datetime")["iv"].dropna().values)
        n = len(hist)
        for lag in [1,2,3]:
            res.loc[grp.index, f"iv_lag{lag}"] = hist[-lag] if n>=lag else np.nan
        for w in ROLLING_WINDOWS:
            win = hist[-w:] if n>=1 else np.array([])
            res.loc[grp.index, f"iv_roll{w}_mean"] = np.mean(win) if len(win)>0 else np.nan
            res.loc[grp.index, f"iv_roll{w}_std"]  = np.std(win)  if len(win)>1 else np.nan
        if n > 0:
            a=2/(EWMA_SPAN+1); ev=hist[0]
            for v in hist[1:]: ev=a*v+(1-a)*ev
            res.loc[grp.index, "iv_ewma"] = ev
            
    for c in temporal_cols:
        res[c] = res[c].fillna(train_medians.get(c, 0.0))
    return res.reset_index()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Single fold execution (Pure, Natural Residual Training)
# ─────────────────────────────────────────────────────────────────────────────
def run_fold(df_long, train_ts, val_ts, fold_seed, base_model):
    train_df = df_long[df_long["datetime"].isin(train_ts)].copy()
    val_df   = df_long[df_long["datetime"].isin(val_ts)].copy()

    train_obs = train_df["iv"].dropna()
    clip_lo   = max(DATA_IV_FLOOR, float(np.percentile(train_obs, CLIP_PCT_LO)))
    clip_hi   = min(DATA_IV_CAP,   float(np.percentile(train_obs, CLIP_PCT_HI)))
    fallback  = float(train_obs.median())

    # VAL HOLDOUT MASK (For scoring only)
    np.random.seed(SEED + fold_seed)
    obs_val_idx  = val_df.index[val_df["iv"].notna()]
    n_holdout    = int(len(obs_val_idx) * HOLDOUT_RATIO)
    holdout_idx  = set(np.random.choice(obs_val_idx, size=n_holdout, replace=False))
    val_df_masked = val_df.copy()
    val_df_masked.loc[list(holdout_idx), "iv"] = np.nan

    train_feat, train_medians, _ = build_temporal_train(train_df)
    train_feat = build_spatial(train_feat)

    # NO MASKING: Fit splines on ALL available training data.
    splines = fit_splines(train_feat, clip_lo, clip_hi)
    train_feat["spline_pred"] = apply_splines(train_feat, splines, fallback, clip_lo, clip_hi)
    
    # Calculate pure organic residuals
    train_feat["residual"] = train_feat["iv"] - train_feat["spline_pred"]

    # Winsorize the target residuals cleanly
    obs_mask  = train_feat["iv"].notna()
    res_vals  = train_feat.loc[obs_mask, "residual"].values
    res_lo    = np.percentile(res_vals, 100-RESIDUAL_WINSOR)
    res_hi    = np.percentile(res_vals, RESIDUAL_WINSOR)
    train_feat.loc[obs_mask, "residual"] = np.clip(res_vals, res_lo, res_hi)

    model = clone(base_model)
    model.fit(train_feat.loc[obs_mask, FEATURE_COLS].values,
              train_feat.loc[obs_mask, "residual"].values)

    train_history = train_df[["datetime","strike","option_type","iv", "row_id"]].copy()
    pred_history  = pd.DataFrame(columns=["datetime","strike","option_type","iv", "row_id"])

    val_sorted_ts = sorted(val_df_masked["datetime"].unique())
    all_preds = []

    for ts in val_sorted_ts:
        ts_rows = val_df_masked[val_df_masked["datetime"]==ts].copy()
        pred_ts_list = sorted(pred_history["datetime"].unique()) if len(pred_history) else []
        recent_pred  = (pred_history[pred_history["datetime"].isin(pred_ts_list[-LAG_WINDOW:])]
                        if pred_ts_list else pred_history)
        combined_history = pd.concat([train_history, recent_pred], ignore_index=True)

        ts_rows = compute_temporal_for_ts(ts_rows, combined_history, train_medians)
        ts_rows = build_spatial(ts_rows)
        
        # Fit baseline strictly on available data in this validation block
        val_ts_splines = fit_splines(ts_rows, clip_lo, clip_hi)
        ts_rows["spline_pred"] = apply_splines(ts_rows, val_ts_splines, fallback, clip_lo, clip_hi)

        raw_res = model.predict(ts_rows[FEATURE_COLS].values)
        ts_rows["residual_pred"] = np.clip(raw_res, res_lo, res_hi) 
        
        ts_rows["iv_pred"] = (ts_rows["spline_pred"] + ts_rows["residual_pred"]).clip(clip_lo, clip_hi)

        for _, row in ts_rows.iterrows():
            if row["row_id"] in holdout_idx: 
                true_iv = val_df.loc[val_df["row_id"] == row["row_id"], "iv"].values[0]
                if not pd.isna(true_iv):
                    all_preds.append({
                        "iv_true": true_iv, "iv_pred": row["iv_pred"], "strike_bucket": row["strike_bucket"],
                    })

        nh = ts_rows[["datetime","strike","option_type","row_id"]].copy()
        nh["iv"] = ts_rows["iv_pred"].values
        pred_history = pd.concat([pred_history, nh], ignore_index=True)

    results = pd.DataFrame(all_preds)
    if len(results) == 0:
        return np.nan, np.nan, np.nan

    y_true, y_pred = results["iv_true"].values, results["iv_pred"].values
    return mean_squared_error(y_true, y_pred), np.sqrt(mean_squared_error(y_true, y_pred)), mean_absolute_error(y_true, y_pred)

In [10]:

# ─────────────────────────────────────────────────────────────────────────────
# Validation Loop
# ─────────────────────────────────────────────────────────────────────────────
unique_times = np.array(sorted(df_long["datetime"].unique()))
tscv         = TimeSeriesSplit(n_splits=N_SPLITS)

print(f"\n{'═'*70}")
print(f"MODEL  | {N_SPLITS} folds | embargo={EMBARGO_STEPS} | holdout={HOLDOUT_RATIO:.0%}")
print(f"{'═'*70}")

model_performance = {}

for model_name, base_model in MODEL_ZOO.items():
    print(f"\nEvaluating: {model_name}")
    fold_mses = []
    
    for fold_i, (tr_idx, vl_idx) in enumerate(tscv.split(unique_times)):
        train_ts = unique_times[tr_idx]
        if EMBARGO_STEPS > 0 and len(train_ts) > EMBARGO_STEPS:
            train_ts = train_ts[:-EMBARGO_STEPS] 
        val_ts = unique_times[vl_idx]

        mse, rmse, mae = run_fold(df_long, train_ts, val_ts, fold_i, base_model)
        fold_mses.append(mse)
        
        mse_str = f"{mse:.6f}" if not np.isnan(mse) else "nan"
        print(f"  Fold {fold_i+1} | train={len(train_ts):4d} | val={len(val_ts):4d} | MSE={mse_str}")
        
    avg_mse = np.nanmean(fold_mses)
    model_performance[model_name] = avg_mse
    print(f"  > {model_name} Average CV MSE: {avg_mse:.6f}")

best_model_name = min(model_performance, key=model_performance.get)
print(f"\n WINNER: {best_model_name} (MSE: {model_performance[best_model_name]:.6f})")
final_base_model = MODEL_ZOO[best_model_name]



══════════════════════════════════════════════════════════════════════
MODEL  | 5 folds | embargo=2 | holdout=30%
══════════════════════════════════════════════════════════════════════

Evaluating: RandomForest
  Fold 1 | train= 163 | val= 162 | MSE=0.000026
  Fold 2 | train= 325 | val= 162 | MSE=0.000042
  Fold 3 | train= 487 | val= 162 | MSE=0.000039
  Fold 4 | train= 649 | val= 162 | MSE=0.000084
  Fold 5 | train= 811 | val= 162 | MSE=0.260266
  > RandomForest Average CV MSE: 0.052091

Evaluating: GradientBoosting
  Fold 1 | train= 163 | val= 162 | MSE=0.000026
  Fold 2 | train= 325 | val= 162 | MSE=0.000041
  Fold 3 | train= 487 | val= 162 | MSE=0.000038
  Fold 4 | train= 649 | val= 162 | MSE=0.000083
  Fold 5 | train= 811 | val= 162 | MSE=0.260236
  > GradientBoosting Average CV MSE: 0.052084

Evaluating: LightGBM
  Fold 1 | train= 163 | val= 162 | MSE=0.000026
  Fold 2 | train= 325 | val= 162 | MSE=0.000042
  Fold 3 | train= 487 | val= 162 | MSE=0.000038
  Fold 4 | train= 649 | 

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
#  Train Final Model on All Data
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print(f"Training Final {best_model_name} on all observed data...")

df_final, final_medians, _ = build_temporal_train(df_long)
df_final = build_spatial(df_final)

all_obs      = df_final["iv"].dropna()
final_clo    = max(DATA_IV_FLOOR, float(np.percentile(all_obs, CLIP_PCT_LO)))
final_chi    = min(DATA_IV_CAP,   float(np.percentile(all_obs, CLIP_PCT_HI)))
final_fb     = float(all_obs.median())

# NO MASKING IN FINAL TRAINING EITHER
train_splines = fit_splines(df_final, final_clo, final_chi)
df_final["spline_pred"] = apply_splines(df_final, train_splines, final_fb, final_clo, final_chi)

df_final["residual"] = df_final["iv"] - df_final["spline_pred"]

obs_mask  = df_final["iv"].notna()
res_vals  = df_final.loc[obs_mask, "residual"].values
fres_lo   = np.percentile(res_vals, 100-RESIDUAL_WINSOR)
fres_hi   = np.percentile(res_vals, RESIDUAL_WINSOR)
df_final.loc[obs_mask, "residual"] = np.clip(res_vals, fres_lo, fres_hi)

final_model = clone(final_base_model)
final_model.fit(df_final.loc[obs_mask, FEATURE_COLS].values,
                df_final.loc[obs_mask, "residual"].values)

# PREDICT MISSING DATA FOR SUBMISSION
best_inference_splines = fit_splines(df_final, final_clo, final_chi)
df_final["spline_pred"] = apply_splines(df_final, best_inference_splines, final_fb, final_clo, final_chi)

raw_res = final_model.predict(df_final[FEATURE_COLS].values)
df_final["residual_pred"] = np.clip(raw_res, fres_lo, fres_hi) 
df_final["iv_pred"] = (df_final["spline_pred"] + df_final["residual_pred"]).clip(final_clo, final_chi)

df_final["iv_filled"] = df_final["iv"].where(df_final["iv"].notna(), df_final["iv_pred"])

# Assertions
missing_mask = df_final["iv"].isna()
pred_vals    = df_final.loc[missing_mask, "iv_filled"]
assert df_final["iv_filled"].isna().sum() == 0
assert (pred_vals >= DATA_IV_FLOOR).all(), f"Min pred={pred_vals.min():.5f} < floor"
assert (pred_vals <= DATA_IV_CAP).all(),   f"Max pred={pred_vals.max():.5f} > cap"
print(f" Predicted {missing_mask.sum()} rows successfully.")



──────────────────────────────────────────────────────────────────────
Training Final GradientBoosting on all observed data...
 Predicted 5460 rows successfully.


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
#  Reconstruct wide filled_dataset.csv
# ─────────────────────────────────────────────────────────────────────────────
pred_map  = df_final.set_index(["datetime","option_col"])["iv_filled"].to_dict()
df_filled = df_original.copy()
fill_count = 0
for col in option_cols:
    for idx in df_filled.index[df_filled[col].isna()]:
        key = (df_filled.loc[idx,"datetime"], col)
        if key in pred_map:
            df_filled.at[idx, col] = pred_map[key]; fill_count += 1

remaining = df_filled[option_cols].isna().sum().sum()
if remaining > 0:
    for col in option_cols:
        med = df_filled[col].median()
        df_filled[col] = df_filled[col].fillna(med if not pd.isna(med) else final_fb)

assert df_filled[option_cols].isna().sum().sum() == 0
df_filled.to_csv(FILLED_PATH, index=False)
print(f"{FILLED_PATH} saved | {fill_count} cells filled")



filled_dataset.csv saved | 5460 cells filled


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
#  Submission
# ─────────────────────────────────────────────────────────────────────────────
SEPARATOR = "||"
def generate_solution(filled_path, output_path):
    original     = pd.read_csv(DATASET_PATH)
    filled       = pd.read_csv(filled_path)
    feature_cols = [c for c in original.columns if c != "datetime"]
    rows = []
    for col in feature_cols:
        was_missing = original[col].isna()
        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            rows.append({"id": uid, "value": filled.loc[idx, col]})
    sol = pd.DataFrame(rows, columns=["id","value"]).sort_values("id").reset_index(drop=True)
    sol.to_csv(output_path, index=False)
    print(f" {output_path} generated ({len(sol)} rows ready for submission)")
    return sol

solution = generate_solution(FILLED_PATH, SUBMISSION_PATH)

 submission.csv generated (5460 rows ready for submission)
